In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
import optuna
import os
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 로드 및 정합성 처리
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

# Train 중복 행 제거 (ID 제외 기준)
dup_cols = [col for col in train.columns if col != 'ID']
train = train.drop_duplicates(subset=dup_cols).reset_index(drop=True)

# 2. 결측치 선행 처리 (팀 가이드 준수)
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

# 3. 팀 공통 파생 변수 생성
def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)

    # 1. 과로 및 생활 리듬
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)

    # 2. 질환 및 유전력
    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)

    # 3. 심혈관 및 신체
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']

    # 4. 대사 및 노화
    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)

    return data

train = add_features(train)
test = add_features(test)

# 4. [신규 전략] 범주형 변수 직접 결합 (Magic Features)
def add_interaction_categories(df):
    data = df.copy()
    data['med_sleep'] = data['medical_history'].astype(str) + '_' + data['sleep_pattern'].astype(str)
    data['med_overwork'] = data['medical_history'].astype(str) + '_' + data['is_overworking'].astype(str)
    data['sleep_overwork'] = data['sleep_pattern'].astype(str) + '_' + data['is_overworking'].astype(str)
    data['med_sleep_overwork'] = data['medical_history'].astype(str) + '_' + data['sleep_pattern'].astype(str) + '_' + data['is_overworking'].astype(str)
    return data

train = add_interaction_categories(train)
test = add_interaction_categories(test)

# 5. Label Encoding (결합 피처 포함)
cat_cols = ['gender', 'activity', 'smoke_status', 'medical_history',
            'family_medical_history', 'sleep_pattern', 'edu_level',
            'med_sleep', 'med_overwork', 'sleep_overwork', 'med_sleep_overwork']

for col in cat_cols:
    le = LabelEncoder()
    # Test 데이터에만 존재하는 결합 범주 처리를 위해 전체 데이터 기준 fit
    le.fit(list(train[col].astype(str)) + list(test[col].astype(str)))
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))
    
    # LightGBM 학습을 위한 category 타입 변환
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score'] # 로그 변환 제거, 원본 유지
x_test = test.drop('ID', axis=1)

# 6. Optuna 하이퍼파라미터 튜닝
def objective(trial):
    params = {
        'objective': 'regression_l1',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'random_state': 42,
        'verbose': -1,
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255, step=16),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
    }
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_mae = []
    
    for tr_idx, val_idx in kf.split(x_train):
        X_tr, X_val = x_train.iloc[tr_idx], x_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr, categorical_feature=cat_cols)
        
        pred = model.predict(X_val)
        cv_mae.append(mean_absolute_error(y_val, pred))
        
    return np.mean(cv_mae)

print("★ Optuna 튜닝 시작 (30회 반복) ★")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)
print(f"\n★ Best Optuna CV MAE: {study.best_value:.4f}")

# 7. 최적 파라미터로 최종 학습 및 예측
best_params = study.best_params
best_params.update({'objective': 'regression_l1', 'metric': 'mae', 'random_state': 42, 'verbose': -1})

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(x_train))
test_preds = np.zeros(len(x_test))

for tr_idx, val_idx in kf.split(x_train):
    X_tr, X_val = x_train.iloc[tr_idx], x_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    
    final_model = lgb.LGBMRegressor(**best_params)
    final_model.fit(X_tr, y_tr, categorical_feature=cat_cols)
    
    oof_preds[val_idx] = final_model.predict(X_val)
    test_preds += final_model.predict(x_test) / kf.n_splits

final_cv_mae = mean_absolute_error(y_train, oof_preds)
print(f"\n★ 최종 모델 자체 점수 (OOF CV MAE): {final_cv_mae:.4f}")

# 8. 예측값 클리핑 및 제출 파일 저장
test_preds = np.clip(test_preds, 0, 1)
sample_submission['stress_score'] = test_preds
submit_path = '../submissions/submit_15_category_concat.csv'
os.makedirs('../submissions', exist_ok=True)
sample_submission.to_csv(submit_path, index=False)
print(f"★ 제출 파일 생성 완료: {submit_path}")